# Working with Files and Data

## pathlib: Treat Paths as Objects

pathlib, available in the standard library, gives you Path objects that work across operating systems and compose naturally.

### Creating and Joining Paths

That / operator is not doing division here. pathlib uses it for path joining, which reads more clearly than os.path.join("data", "documents").

In [1]:
from pathlib import Path

# Create a Path object
data_dir = Path("data")
documents_dir = data_dir / "documents"  # The / operator joins paths
config_file = data_dir / "config.json"

print(documents_dir)  # data/documents
print(config_file)    # data/config.json

data\documents
data\config.json


### Useful Path Properties

In [2]:
from pathlib import Path

path = Path("/home/user/datasets/train.csv")

print(path.stem)    # train         (filename without extension)
print(path.suffix)  # .csv          (file extension)
print(path.parent)  # /home/user/datasets  (parent directory)
print(path.name)    # train.csv     (full filename)

train
.csv
\home\user\datasets
train.csv


### Reading and Writing Files

For small files, pathlib gives you one-liner convenience methods to work with files

For large files, stream with open() instead of reading the whole file into memory.


In [3]:
from pathlib import Path

# Write text to a file
config_path = Path("config.json")
config_path.write_text('{"model": "example-model", "temperature": 0.7}', encoding="utf-8")

# Read it back
content = config_path.read_text(encoding="utf-8")
print(content)  # {"model": "example-model", "temperature": 0.7}

{"model": "example-model", "temperature": 0.7}


### Checking Existence and Creating Directories

The parents=True flag creates missing parent directories too. Without it, Python raises an error if a parent directory does not exist. The exist_ok=True flag prevents an error when the directory is already there

In [4]:
from pathlib import Path

data_dir = Path("output/results")

# Check if path exists
if not data_dir.exists():
    data_dir.mkdir(parents=True, exist_ok=True)  # Creates all parent dirs

# Check what kind of path it is
print(data_dir.is_dir())   # True
print(data_dir.is_file())  # False

True
False


### Finding Files with glob

In [5]:
from pathlib import Path

data_dir = Path("documents")

# Find all PDFs in the directory
pdfs = list(data_dir.glob("*.pdf"))

# Find all PDFs recursively (including subdirectories)
all_pdfs = list(data_dir.rglob("*.pdf"))

# Find all JSON files recursively
json_files = list(data_dir.rglob("*.json"))

print(f"Found {len(all_pdfs)} PDFs and {len(json_files)} JSON files")

Found 0 PDFs and 0 JSON files


## Context Managers: Safe File Handling

In [8]:
f = open("data.txt", "r")
content = f.read()
print(content)
f.close()

hello world! dummy text


If f.read() raises an exception, f.close() never runs. The file handle stays open. In a long-running service or batch job, repeated leaks can hit the operating system's limit on open file descriptors.

The with statement fixes this by guaranteeing cleanup

In [9]:
with open("data.txt", "r") as f:
    content = f.read()
# f.close() is called automatically, even if an exception occurs
print(content)

hello world! dummy text


### Writing Your Own Context Manager

Everything before yield runs when entering the with block. Everything after it runs when exiting. Wrapping yield in try/finally guarantees the cleanup code runs even if the block raises an exception.

In [10]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label):
    start = time.time()
    try:
        yield
    finally:
        # The 'finally' block runs on exit, even if the block raised
        elapsed = time.time() - start
        print(f"{label}: {elapsed:.2f} seconds")

with timer("PDF processing"):
    time.sleep(0.5)
# Prints: PDF processing: 0.5 seconds

PDF processing: 0.50 seconds


## Reading and Writing JSON

### The Basics: load, dump, loads, dumps

In [ ]:
import json

# String operations (the 's' stands for 'string')
data = json.loads('{"model": "example-model", "temperature": 0.7}')  # string -> dict
text = json.dumps(data)  # dict -> string

# File operations (no 's')
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(data, f)  # dict -> file

with open("config.json", "r", encoding="utf-8") as f:
    data = json.load(f)  # file -> dict

{'model': 'example-model', 'temperature': 0.7}


### Pretty Printing

In [24]:
import json

data = {"model": "example-model", "parameters": {"temperature": 0.7, "max_tokens": 1000}}

# Compact (good for APIs, bad for humans)
print(json.dumps(data))
# {"model": "example-model", "parameters": {"temperature": 0.7, "max_tokens": 1000}}

# Pretty (good for config files and debugging)
print(json.dumps(data, indent=2))
# {
#   "model": "example-model",
#   "parameters": {
#     "temperature": 0.7,
#     "max_tokens": 1000
#   }
# }

{"model": "example-model", "parameters": {"temperature": 0.7, "max_tokens": 1000}}
{
  "model": "example-model",
  "parameters": {
    "temperature": 0.7,
    "max_tokens": 1000
  }
}


### Handling Encoding Issues

The ensure_ascii=False flag tells json.dumps to write Unicode characters directly instead of escaping them as \uXXXX. This keeps the file readable for humans

In [ ]:
from pathlib import Path
import json

# Safe approach for international text
data = {"text": "Caf\u00e9 AI \u2014 \u65e5\u672c\u8a9e\u30c6\u30b9\u30c8"}

Path("output.json").write_text(
    json.dumps(data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# using the 'with' statement to write JSON data to a file
with open("output.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(data, ensure_ascii=False, indent=2))

### JSONL: JSON Lines

AI workflows often use the JSONL format, where each line is a separate JSON object. This format is common for datasets, evaluation sets, and log files

In [35]:
import json

# Writing JSONL
records = [
    {"prompt": "What is Python?", "completion": "A programming language."},
    {"prompt": "What is RAG?", "completion": "Retrieval-Augmented Generation."},
]

with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Reading JSONL
with open("dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            print(record)

{'prompt': 'What is Python?', 'completion': 'A programming language.'}
{'prompt': 'What is RAG?', 'completion': 'Retrieval-Augmented Generation.'}


## Reading and Writing CSV

### DictReader and DictWriter

One thing to watch: DictReader returns all values as strings, even numbers. Cast them yourself, for example float(row["accuracy"]), if you need to do math

In [36]:
import csv

# Writing CSV
results = [
    {"model": "model-a", "accuracy": 0.92, "latency_ms": 450},
    {"model": "model-b", "accuracy": 0.89, "latency_ms": 380},
    {"model": "model-c", "accuracy": 0.78, "latency_ms": 120},
]

with open("benchmark.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["model", "accuracy", "latency_ms"])
    writer.writeheader()
    writer.writerows(results)

# Reading CSV
with open("benchmark.csv", "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['model']}: accuracy={row['accuracy']}")

model-a: accuracy=0.92
model-b: accuracy=0.89
model-c: accuracy=0.78


## Making HTTP Requests with requests

### GET and POST Requests

The json= parameter serializes your dictionary to JSON and sends the request with a JSON content type. You could also use data=json.dumps(payload), but json= is clearer for normal API calls.

In [ ]:
import requests

# Simple GET request
response = requests.get("https://api.example.com/models")
print(response.status_code)  # 200
data = response.json()       # Parse JSON response body

# POST request with JSON body (typical for LLM APIs)
response = requests.post(
    "https://api.example.com/v1/chat/completions",
    headers={
        "Authorization": "Bearer sk-your-api-key",
        "Content-Type": "application/json",
    },
    json={
        "model": "example-model",
        "messages": [{"role": "user", "content": "Hello!"}],
    },
)

### Response Handling


In [ ]:
response = requests.get("https://api.example.com/data")

# Status code
print(response.status_code)  # 200, 404, 500, etc.

# Raise an exception for 4xx/5xx responses
response.raise_for_status()  # Raises requests.HTTPError if status >= 400

# Response body
data = response.json()       # Parse as JSON
text = response.text         # Raw text
raw = response.content       # Raw bytes (useful for binary data like images)

The .raise_for_status() method is important in real code. Without it, a failed API call still returns a response object, and your program may not fail until later when it tries to use missing or malformed data. Also remember that .json() can raise an error if the response body is not valid JSON.

### Timeouts: Do Not Skip These

In [ ]:
# Always set a timeout
response = requests.get(
    "https://api.example.com/data",
    timeout=30  # seconds
)

# You can also set connect and read timeouts separately
response = requests.get(
    "https://api.example.com/data",
    timeout=(5, 30)  # 5 seconds to connect, 30 seconds to read
)

## httpx: Sync and Async HTTP

In [ ]:
import httpx

# Synchronous API with a familiar shape
response = httpx.get("https://api.example.com/data", timeout=30)
data = response.json()

# POST with headers and a JSON body
response = httpx.post(
    "https://api.example.com/v1/embeddings",
    headers={"Authorization": "Bearer sk-your-key"},
    json={"input": "Hello world", "model": "text-embedding-3-small"},
    timeout=30,
)

In [ ]:
import httpx
import asyncio

async def fetch_embeddings(texts):
    async with httpx.AsyncClient(timeout=30) as client:
        tasks = [
            client.post(
                "https://api.example.com/v1/embeddings",
                headers={"Authorization": "Bearer sk-your-key"},
                json={"input": text, "model": "text-embedding-3-small"},
            )
            for text in texts
        ]
        responses = await asyncio.gather(*tasks)
        for response in responses:
            response.raise_for_status()
        return [r.json() for r in responses]

requests is fine for straightforward synchronous scripts. httpx.AsyncClient is a better fit when one workflow needs many concurrent network calls.

## Reading PDFs


### PyMuPDF (fitz)

In [ ]:
%pip install pymupdf
import fitz  # pip install pymupdf

def extract_text_from_pdf(pdf_path):
    """Extract text from all pages of a PDF."""
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()
            pages.append({
                "page": page_num,
                "text": text.strip(),
            })
    return pages

# Usage
pages = extract_text_from_pdf("research_paper.pdf")
for p in pages:
    print(f"Page {p['page']}: {len(p['text'])} characters")

For scanned PDFs, you need OCR, which stands for Optical Character Recognition.

Libraries like pytesseract or cloud OCR services can handle this, but OCR brings its own trade-offs around accuracy, cost, latency, and privacy

In [ ]:
pages = extract_text_from_pdf("scanned_document.pdf")
non_empty = [p for p in pages if len(p["text"]) > 50]

if len(non_empty) < len(pages) * 0.5:
    print("Warning: This PDF may be scanned. OCR may be needed.")

## Parsing HTML with BeautifulSoup

In [ ]:
from bs4 import BeautifulSoup  # pip install beautifulsoup4
import requests

# Fetch a web page
response = requests.get("https://example.com/article", timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# Extract the page title
title_tag = soup.find("title")
title = title_tag.get_text(strip=True) if title_tag else None

# Extract all paragraph text
paragraphs = [p.text.strip() for p in soup.find_all("p")]

# Extract text from a specific div
main_content = soup.find("div", class_="article-body")
if main_content:
    article_text = main_content.get_text(separator="\n", strip=True)

The get_text(separator="\n", strip=True) call is useful because it extracts text from an element and its children, joins pieces with newlines, and strips extra whitespace

### Extracting Links and Metadata

In [ ]:
# Extract all links
links = []
for a_tag in soup.find_all("a", href=True):
    links.append({
        "text": a_tag.text.strip(),
        "url": a_tag["href"],
    })

# Extract meta description
meta = soup.find("meta", attrs={"name": "description"})
description = meta["content"] if meta else None

## Working with Large Files

### Line-by-Line Reading

In [ ]:
import json

def iter_jsonl(file_path):
    """Yield JSONL records without loading the whole file into memory."""
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
            record = json.loads(line)
            if (i + 1) % 100_000 == 0:
                print(f"Processed {i + 1:,} records...")
            yield record

for record in iter_jsonl("large_dataset.jsonl"):
    preview = record["text"][:100]
    # Process or write each result here, instead of accumulating all previews.

### Chunked Reading for Binary Files

In [ ]:
def read_in_chunks(file_path, chunk_size=8192):
    """Read a file in chunks of chunk_size bytes."""
    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            yield chunk

# Usage: calculate file hash without loading into memory
import hashlib

hasher = hashlib.sha256()
for chunk in read_in_chunks("large_model.bin"):
    hasher.update(chunk)
print(f"SHA-256: {hasher.hexdigest()}")

## Putting It All Together: A Data Ingestion Pipeline

In [ ]:
import json
import fitz
import requests
from pathlib import Path
from bs4 import BeautifulSoup

def load_pdf(path):
    """Extract text from a PDF file."""
    texts = []
    with fitz.open(path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text().strip()
            if text:
                texts.append({
                    "source": str(path),
                    "page": page_number,
                    "text": text,
                })
    return texts

def load_json(path):
    """Load records from a JSON file."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(data, list):
        return [
            {"source": str(path), "text": json.dumps(item, ensure_ascii=False)}
            for item in data
        ]
    return [{"source": str(path), "text": json.dumps(data, ensure_ascii=False)}]

def load_from_url(url):
    """Fetch and parse a web page."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator="\n", strip=True)
    return [{"source": url, "text": text}]

def ingest_documents(input_dir, output_path):
    """Ingest all documents from a directory into a JSONL file."""
    input_dir = Path(input_dir)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    total_records = 0

    with open(output_path, "w", encoding="utf-8") as f:
        for path in input_dir.rglob("*"):
            if not path.is_file():
                continue

            suffix = path.suffix.lower()

            if suffix == ".pdf":
                loader = load_pdf
            elif suffix == ".json":
                loader = load_json
            else:
                continue

            try:
                for record in loader(path):
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    total_records += 1
            except Exception as exc:
                print(f"Skipped {path}: {exc}")

    print(f"Ingested {total_records} records to {output_path}")

# Usage
ingest_documents("data/raw_documents", "data/processed/corpus.jsonl")